# How to Manage a Portfolio of Strategies

A portfolio of strategies is more than a list of backtests. This notebook goes through the steps in order: why diversification works, how to bring every strategy to the same risk (volatility targeting, with and without a stop loss), how to assign weights, why optimizing the weights is another way to overfit, and how to check an allocation method with a walk-forward. All data is **synthetic**: twelve daily return series with **different true Sharpe ratios** (0.2 to 1.0), different win-rate profiles and **correlations that drift over time**, so every result can be checked against the truth.

**Contents**

1. Twelve synthetic strategies: different edges, different styles
2. Why diversify
3. Diminishing returns
4. Raw equity curves are not comparable
5. Volatility targeting
6. Case 1: risk per trade with a stop loss
7. Case 2: no stop loss, volatility-based sizing
8. Weighting methods
9. Equal weight + correlation
10. Why optimizing the portfolio overfits
11. Walk-forward on the portfolio optimization
12. The hidden trap: in-sample strategy returns
13. Balancing strategy types
14. Balancing instruments and asset classes
15. Correlation is not stable
16. Practical issues: minimum N and lot rounding
17. Choosing the target volatility

Requirements: `pip install numpy pandas scipy plotly`

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.optimize import brentq, minimize
from scipy.stats import norm
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

pio.templates.default = "plotly_dark"
# if charts don't show up (some VS Code / JupyterLab versions): pio.renderers.default = "notebook"

def plot_lines(series, title, ytitle=None, xtitle=None, logy=False, height=700):
    fig = go.Figure([go.Scatter(x=s.index, y=s, name=name, line_width=4) for name, s in series.items()])
    fig.update_layout(title=title, xaxis_title=xtitle, yaxis_title=ytitle, height=height, hovermode="x unified")
    if logy:
        fig.update_yaxes(type="log")
    fig.show()

def plot_bars(df, title, ytitle=None, barmode="group", height=650):
    fig = go.Figure([go.Bar(x=df.index, y=df[c], name=c) for c in df])
    fig.update_layout(title=title, yaxis_title=ytitle, barmode=barmode, height=height)
    fig.show()

In [2]:
ANN = 252
TARGET_VOL = 0.10          # annual volatility target of each strategy and of the portfolio
MEAN_SR = 0.6              # average true Sharpe ratio (each strategy has its own, see STRATS)
YEARS = 12
SEED = 42

def ann_vol(r):
    return r.std() * np.sqrt(ANN)

def sharpe(r):
    return r.mean() / r.std() * np.sqrt(ANN)

def max_dd(r):
    eq = r.cumsum()
    return (eq - eq.cummax()).min()

def metrics(r):
    return pd.Series({"Return": r.mean() * ANN, "Vol": ann_vol(r), "Sharpe": sharpe(r), "MaxDD": max_dd(r)})

FMT = {"Return": "{:.1%}", "Vol": "{:.1%}", "Sharpe": "{:.2f}", "MaxDD": "{:.1%}"}

Returns are **daily, additive** (equity = cumulative sum of returns, no compounding): with a constant risk target this is how a vol-targeted account behaves, and it keeps every curve on the same scale.

## 1. Twelve synthetic strategies: different edges, different styles

Each strategy is defined by its **style** and by its own **edge**:

- **exposure**: share of days in the market (the other days return 0);
- **win rate**: share of positive days among the days in the market;
- wins and losses are exponential, and their sizes are chosen so that each strategy has its own **true Sharpe ratio**, between 0.24 and 0.94 (average 0.59). The Sharpe measured on the 12 years of data is noisy, as in real life.

Three families:
- **Trend following**: low win rate (35–40%), few large wins, many small losses → positive skew;
- **Mean reversion**: high win rate (62–68%), many small wins, few large losses → negative skew;
- **Other** (seasonality, carry): in between.

Correlations come from a Gaussian copula: same family, same asset class and same instrument add correlation, trend and mean reversion start slightly negative, but the shared instruments and asset classes bring them to about zero on average. On top of this structure, all pairwise correlations drift slowly (they shift every quarter, within roughly ±0.15), and three **stress periods** of three months push them up (used in section 15).

In [3]:
STRATS = pd.DataFrame([
    # name           family     instrument asset class  win rate exposure raw vol true SR
    ("TF_US500",     "trend",   "US500",   "indices",   0.38,    0.70,    0.14,   0.75),
    ("TF_XAUUSD",    "trend",   "XAUUSD",  "metals",    0.36,    0.65,    0.25,   0.24),
    ("TF_GBPJPY",    "trend",   "GBPJPY",  "fx",        0.40,    0.75,    0.09,   0.38),
    ("TF_WTI",       "trend",   "WTI",     "energy",    0.35,    0.60,    0.20,   0.35),
    ("TF_CORN",      "trend",   "CORN",    "agri",      0.37,    0.65,    0.12,   0.34),
    ("MR_US500",     "meanrev", "US500",   "indices",   0.66,    0.35,    0.08,   0.85),
    ("MR_NAS100",    "meanrev", "NAS100",  "indices",   0.68,    0.30,    0.11,   0.94),
    ("MR_EURUSD",    "meanrev", "EURUSD",  "fx",        0.62,    0.40,    0.05,   0.42),
    ("MR_GER40",     "meanrev", "GER40",   "indices",   0.65,    0.35,    0.07,   0.86),
    ("SEAS_US500",   "other",   "US500",   "indices",   0.55,    0.25,    0.06,   0.91),
    ("SEAS_NATGAS",  "other",   "NATGAS",  "energy",    0.50,    0.30,    0.18,   0.61),
    ("CARRY_AUDJPY", "other",   "AUDJPY",  "fx",        0.56,    0.90,    0.10,   0.40),
], columns=["name", "family", "instrument", "asset_class", "win_rate", "exposure", "raw_vol", "sr"]).set_index("name")
STRATS

,family,instrument,asset_class,win_rate,exposure,raw_vol,sr
name,,,,,,,
TF_US500,trend,US500,indices,0.38,0.70,0.14,0.75
TF_XAUUSD,trend,XAUUSD,metals,0.36,0.65,0.25,0.24
TF_GBPJPY,trend,GBPJPY,fx,0.40,0.75,0.09,0.38
TF_WTI,trend,WTI,energy,0.35,0.60,0.20,0.35
TF_CORN,trend,CORN,agri,0.37,0.65,0.12,0.34
MR_US500,meanrev,US500,indices,0.66,0.35,0.08,0.85
MR_NAS100,meanrev,NAS100,indices,0.68,0.30,0.11,0.94
MR_EURUSD,meanrev,EURUSD,fx,0.62,0.40,0.05,0.42
MR_GER40,meanrev,GER40,indices,0.65,0.35,0.07,0.86


In [4]:
def target_corr(meta):
    fam, inst, cls = meta.family.values, meta.instrument.values, meta.asset_class.values
    n = len(meta)
    C = np.full((n, n), 0.05)
    for i in range(n):
        for j in range(n):
            if fam[i] == fam[j]:
                C[i, j] = {"trend": 0.30, "meanrev": 0.25, "other": 0.05}[fam[i]]
            elif {fam[i], fam[j]} == {"trend", "meanrev"}:
                C[i, j] = -0.10
            C[i, j] += 0.10 * (cls[i] == cls[j]) + 0.20 * (inst[i] == inst[j])
    np.fill_diagonal(C, 1)
    return C

def nearest_psd(C):
    vals, vecs = np.linalg.eigh(C)
    C = vecs @ np.diag(np.clip(vals, 1e-6, None)) @ vecs.T
    d = np.sqrt(np.diag(C))
    return C / np.outer(d, d)

def marginal_sample(win_rate, exposure, size, rng, sr=MEAN_SR):
    # unit-scale returns: exponential wins (mean a) and losses (mean 1), zero when out of the market
    def daily_sr(a):
        m = exposure * (win_rate * a - (1 - win_rate))
        m2 = 2 * exposure * (win_rate * a ** 2 + (1 - win_rate))
        return m / np.sqrt(m2 - m ** 2)
    a = brentq(lambda a: daily_sr(a) - sr / np.sqrt(ANN), (1 - win_rate) / win_rate, 50)
    active, win = rng.random(size) < exposure, rng.random(size) < win_rate
    return np.where(active, np.where(win, rng.exponential(a, size), -rng.exponential(1, size)), 0.0)

def exact_sharpe(r, target=MEAN_SR):
    # scale the winning days so the in-sample Sharpe is exactly the target (win rate unchanged)
    f = lambda g: sharpe(np.where(r > 0, r * g, r)) - target
    return np.where(r > 0, r * brentq(f, 0.1, 10), r)

def make_strategies(meta, sr=MEAN_SR, years=YEARS, stress=((0.20, 63), (0.55, 63), (0.85, 63)), seed=SEED, exact=False, drift=0.08):
    rng = np.random.default_rng(seed)
    n_days, n = ANN * years, len(meta)
    dates = pd.bdate_range("2014-01-01", periods=n_days)
    C = nearest_psd(target_corr(meta))
    C_stress = nearest_psd(np.where(np.eye(n) == 1, 1, np.minimum(C + 0.45, 0.90)))
    in_stress = np.zeros(n_days, bool)
    for start, length in stress:
        in_stress[int(start * n_days): int(start * n_days) + length] = True
    z = rng.standard_normal((n_days, n))
    # slowly moving correlations: every quarter all pairwise correlations shift by d (AR(1), std ~ 1.7 x drift)
    d = np.zeros(n_days // 63 + 1)
    for q in range(1, len(d)):
        d[q] = 0.8 * d[q - 1] + rng.normal(0, drift)
    for q in range(len(d)):
        sl = slice(q * 63, (q + 1) * 63)
        Cq = nearest_psd(np.where(np.eye(n) == 1, 1, np.clip(C + d[q], -0.5, 0.95))) if drift else C
        z[sl] = z[sl] @ np.linalg.cholesky(Cq).T
    z[in_stress] = rng.standard_normal((in_stress.sum(), n)) @ np.linalg.cholesky(C_stress).T
    u = norm.cdf(z)                                             # Gaussian copula
    out = {}
    for k, (name, row) in enumerate(meta.iterrows()):
        pool = np.sort(marginal_sample(row.win_rate, row.exposure, 200_000, rng, row.get("sr", sr)))
        r = pool[np.minimum((u[:, k] * len(pool)).astype(int), len(pool) - 1)]   # quantile mapping
        r = exact_sharpe(r, row.get("sr", sr)) if exact else r
        out[name] = r / r.std() * row.raw_vol / np.sqrt(ANN)   # raw (unnormalized) volatility
    return pd.DataFrame(out, index=dates), pd.Series(in_stress, index=dates)

raw, stress = make_strategies(STRATS)

def profile(r):
    on = r[r != 0]
    return pd.Series({"Sharpe": sharpe(r), "Vol": ann_vol(r), "Exposure": len(on) / len(r),
                      "Win rate": (on > 0).mean(), "Avg win / avg loss": on[on > 0].mean() / -on[on < 0].mean(),
                      "Skew": r.skew()})

pd.DataFrame({c: profile(raw[c]) for c in raw}).T.style.format(
    {"Sharpe": "{:.2f}", "Vol": "{:.1%}", "Exposure": "{:.0%}", "Win rate": "{:.0%}",
     "Avg win / avg loss": "{:.2f}", "Skew": "{:+.2f}"})

,Sharpe,Vol,Exposure,Win rate,Avg win / avg loss,Skew
TF_US500,0.86,14.0%,68%,38%,1.99,+1.65
TF_XAUUSD,0.86,25.0%,65%,37%,2.03,+1.64
TF_GBPJPY,0.46,9.0%,76%,39%,1.72,+1.43
TF_WTI,0.64,20.0%,60%,38%,1.93,+1.92
TF_CORN,0.57,12.0%,66%,37%,1.95,+1.98
MR_US500,0.78,8.0%,35%,67%,0.63,-1.81
MR_NAS100,1.33,11.0%,30%,67%,0.77,-1.00
MR_EURUSD,0.24,5.0%,40%,61%,0.67,-1.10
MR_GER40,0.92,7.0%,34%,66%,0.68,-1.27
SEAS_US500,0.99,6.0%,25%,56%,1.10,+0.66


Sharpe ratios from 0.24 to 1.33 in this sample (the true ones are 0.24 to 0.94: the difference is sampling noise), very different shapes: trend following wins on about 4 days out of 10, with wins about twice as large as losses; mean reversion wins on 2 days out of 3, with losses about 1.5 times as large as wins. Below, the distribution of daily returns on the days in the market (each series scaled to unit volatility to compare shapes).

In [5]:
fig = go.Figure()
for name in ["TF_XAUUSD", "MR_NAS100", "CARRY_AUDJPY"]:
    on = raw[name][raw[name] != 0]
    fig.add_histogram(x=on / on.std(), name=f"{name} (win rate {(on > 0).mean():.0%})", opacity=0.6,
                      histnorm="probability", xbins=dict(start=-6, end=6, size=0.1))
fig.update_layout(title="Daily returns on active days: different Sharpe, different styles", barmode="overlay",
                  xaxis_title="return / std", yaxis_title="frequency", height=650).show()

In [6]:
plot_lines({c: raw[c].cumsum() for c in raw}, "Raw equity curves (synthetic, additive)", ytitle="cumulative return")

Correlations are part of the data generator, so we can look at them right away. Left: the correlation matrix of the daily returns over the whole sample. Strategies in the same family (average 0.37 among trend following, 0.27 among mean reversion), on the same instrument or in the same asset class are the most correlated; trend following against mean reversion is about zero on average.

In [7]:
corr_raw = raw.corr()
fig = go.Figure(go.Heatmap(z=corr_raw.values, x=corr_raw.columns, y=corr_raw.columns, zmin=-1, zmax=1, zmid=0,
                           colorscale="RdBu_r", text=corr_raw.round(2).values, texttemplate="%{text}", colorbar_title="corr"))
fig.update_layout(title="Correlation matrix of daily returns (full sample)", height=750, yaxis_autorange="reversed")
fig.show()

The full-sample number hides how correlation moves. Rolling 3-month correlation of four pairs: same instrument, same family, same asset class, and two unrelated strategies. The shaded areas are the three stress periods: inside them the correlations jump (for example 0.52 against 0.35 for the two trend strategies), and outside they wander by several tenths even though the full-sample value is a single number.

In [8]:
PAIRS = [("TF_US500", "MR_US500"), ("TF_US500", "TF_WTI"), ("MR_US500", "MR_NAS100"), ("TF_CORN", "CARRY_AUDJPY")]
labels = ["same instrument", "same family", "same asset class", "unrelated"]
series = {f"{a} / {b} ({lab})": raw[a].rolling(63).corr(raw[b]).dropna() for (a, b), lab in zip(PAIRS, labels)}
fig = go.Figure([go.Scatter(x=s.index, y=s, name=name, line_width=3) for name, s in series.items()])
blocks = stress.astype(int).diff().fillna(0)
for a, b in zip(stress.index[blocks == 1], stress.index[blocks == -1]):
    fig.add_vrect(x0=a, x1=b, fillcolor="red", opacity=0.2, line_width=0)
fig.update_layout(title="Rolling 3-month pairwise correlation", yaxis_title="correlation", height=650, hovermode="x unified")
fig.show()

## 2. Why diversify

Equal weights, equal volatility, average correlation $\bar\rho$:

$$SR_p = SR \cdot \sqrt{\frac{N}{1 + (N-1)\bar\rho}} \xrightarrow{N \to \infty} \frac{SR}{\sqrt{\bar\rho}}$$

The theory for $SR = 0.5$, then the same check on our strategies: for each $N$, the average Sharpe of 300 random portfolios of $N$ strategies (vol-targeted, equal weight).

In [9]:
def sr_portfolio(sr, n, rho):
    return sr * np.sqrt(n / (1 + (n - 1) * rho))

ns = np.arange(1, 51)
theory = pd.DataFrame({f"ρ = {rho}": sr_portfolio(0.5, ns, rho) for rho in (0.0, 0.2, 0.5)}, index=ns)
theory.loc[[1, 10]].round(2).T.assign(**{"N → ∞": [np.inf, 0.5 / np.sqrt(0.2), 0.5 / np.sqrt(0.5)]}).round(2)

,1,10,N → ∞
ρ = 0.0,0.5,1.58,inf
ρ = 0.2,0.5,0.94,1.12
ρ = 0.5,0.5,0.67,0.71


In [10]:
vt = raw * (TARGET_VOL / ann_vol(raw))                    # vol-targeted (section 5)
rng = np.random.default_rng(1)
emp = pd.Series({n: np.mean([sharpe(vt[rng.choice(vt.columns, n, replace=False)].mean(axis=1)) for _ in range(300)])
                 for n in range(1, vt.shape[1] + 1)})

corr = vt.corr().values
rho_bar = corr[np.triu_indices(len(corr), 1)].mean()

fig = go.Figure()
for c in theory:
    fig.add_scatter(x=theory.index, y=theory[c], name=f"theory, SR 0.5, {c}", line_width=4)
fig.add_scatter(x=emp.index, y=emp, name=f"our strategies (average SR {sharpe(raw).mean():.2f}, ρ̄ = {rho_bar:.2f})", mode="lines+markers", line_width=4)
fig.add_scatter(x=emp.index, y=sr_portfolio(sharpe(raw).mean(), emp.index.values, rho_bar), name="theory with our ρ̄",
                line=dict(dash="dot", width=3))
fig.update_layout(title="Portfolio Sharpe vs number of strategies", xaxis_title="N strategies",
                  yaxis_title="Sharpe ratio", yaxis_range=[0, 3], height=700).show()

The empirical points follow the formula with our measured average correlation and the average Sharpe measured on these strategies (0.74, higher than the true 0.59 because of sampling luck). With $\rho = 0.2$ the curve flattens near $0.5/\sqrt{0.2} \approx 1.12$: correlation sets the ceiling, not the number of strategies.

## 3. Diminishing returns

The **diversification multiplier** is how much the Sharpe ratio grows when you combine $N$ strategies: $DM = \sqrt{N / (1 + (N-1)\bar\rho)}$.

In [11]:
def div_mult(n, rho):
    return np.sqrt(n / (1 + (n - 1) * rho))

print("DM at ρ = 0.2:", {n: round(div_mult(n, 0.2), 2) for n in (1, 2, 5, 10, 20, 50)}, "| N → ∞:", round(1 / np.sqrt(0.2), 2))

dm = pd.DataFrame({f"ρ = {rho}": div_mult(ns, rho) for rho in (0.1, 0.2, 0.3, 0.5)}, index=ns)
fig = go.Figure([go.Scatter(x=dm.index, y=dm[c], name=c, line_width=4) for c in dm])
fig.add_vrect(x0=10, x1=15, fillcolor="green", opacity=0.15, line_width=0, annotation_text="10–15 strategies")
fig.update_layout(title="Diversification multiplier: most of the gain comes early", xaxis_title="N strategies",
                  yaxis_title="DM", height=700).show()

DM at ρ = 0.2: {1: np.float64(1.0), 2: np.float64(1.29), 5: np.float64(1.67), 10: np.float64(1.89), 20: np.float64(2.04), 50: np.float64(2.15)} | N → ∞: 2.24


Adding one more strategy to an existing portfolio: its benefit depends on its correlation with what you already have. Marginal Sharpe of a 11th strategy (same SR) added to 10 strategies with $\bar\rho = 0.2$, as a function of its own average correlation with them:

In [12]:
def sr_add_one(sr, n, rho, rho_new):
    # n strategies with average correlation rho, plus one with average correlation rho_new to them
    var = (n + n * (n - 1) * rho + 1 + 2 * n * rho_new) / (n + 1) ** 2
    return sr / np.sqrt(var)

rho_new = np.linspace(-0.2, 0.8, 51)
base = sr_portfolio(0.5, 10, 0.2)
gain = pd.Series(sr_add_one(0.5, 10, 0.2, rho_new) / base - 1, index=rho_new)
plot_lines({"Sharpe gain from the 11th strategy": gain * 100}, "Value of a new strategy depends on its correlation",
           ytitle="Sharpe change (%)", xtitle="correlation of the new strategy with the portfolio", height=600)
print(f"rho_new = 0.0: {gain.iloc[np.abs(rho_new - 0).argmin()]:+.1%} | rho_new = 0.6: {gain.iloc[np.abs(rho_new - 0.6).argmin()]:+.1%}")

rho_new = 0.0: +8.1% | rho_new = 0.6: -9.1%


## 4. Raw equity curves are not comparable

Three of our strategies, combined with **equal capital**, without any risk normalization. The bar chart shows how much each one contributes to the portfolio variance:

$$RC_i = \frac{w_i (\Sigma w)_i}{w^\top \Sigma w}$$

In [13]:
def risk_contrib(w, cov):
    w = np.asarray(w, float)
    rc = w * (cov @ w)
    return rc / rc.sum()

trio = ["MR_GER40", "TF_XAUUSD", "TF_CORN"]
plot_lines({f"{c} (vol {ann_vol(raw[c]):.0%})": raw[c].cumsum() for c in trio},
           "Three raw strategies: same Sharpe, very different risk", ytitle="cumulative return")

rc = pd.DataFrame({"capital weight": np.full(3, 1 / 3),
                   "risk contribution": risk_contrib(np.full(3, 1 / 3), raw[trio].cov().values)}, index=trio)
plot_bars(rc * 100, "Equal capital ≠ equal risk", ytitle="% of total")
rc.style.format("{:.0%}")

,capital weight,risk contribution
MR_GER40,33%,5%
TF_XAUUSD,33%,71%
TF_CORN,33%,24%


With equal capital, gold trend following (25% vol) carries most of the portfolio risk. Three strategies on paper, almost one in practice.

## 5. Volatility targeting

Scale each strategy so that, standalone, it runs at the same annual volatility:

$$k_i = \frac{\sigma_{target}}{\sigma_i}, \qquad \sigma_i = \text{std}(r_{daily}) \cdot \sqrt{252}$$

In [14]:
k = TARGET_VOL / ann_vol(raw)
vt = raw * k

plot_lines({f"{c} (k = {k[c]:.2f})": vt[c].cumsum() for c in trio},
           f"Same three strategies, each scaled to {TARGET_VOL:.0%} annual vol", ytitle="cumulative return")

rc_vt = pd.DataFrame({"raw, equal capital": risk_contrib(np.full(3, 1 / 3), raw[trio].cov().values),
                      "vol-targeted, equal weight": risk_contrib(np.full(3, 1 / 3), vt[trio].cov().values)}, index=trio)
plot_bars(rc_vt * 100, "Risk contribution before and after vol targeting", ytitle="% of portfolio risk")
pd.DataFrame({"raw vol": ann_vol(raw), "k": k, "vol after": ann_vol(vt)}).T.round(3)

,TF_US500,TF_XAUUSD,TF_GBPJPY,TF_WTI,TF_CORN,MR_US500,MR_NAS100,MR_EURUSD,MR_GER40,SEAS_US500,SEAS_NATGAS,CARRY_AUDJPY
raw vol,0.140,0.25,0.090,0.2,0.120,0.08,0.110,0.05,0.070,0.060,0.180,0.1
k,0.714,0.40,1.111,0.5,0.833,1.25,0.909,2.00,1.428,1.666,0.555,1.0
vol after,0.100,0.10,0.100,0.1,0.100,0.10,0.100,0.10,0.100,0.100,0.100,0.1


After scaling, the risk contributions are roughly one third each (27% / 36% / 36%, not exactly because the correlations differ). From here on, a weight means a share of the **risk budget**, not of the capital. All the following sections use `vt`, the vol-targeted returns.

## 6. Case 1: risk per trade with a stop loss

With a fixed risk per trade $r$ (fraction of equity lost at the stop), the annual volatility depends on the number of trades and on the dispersion of the R-multiples:

$$\sigma_{annual} \approx r \cdot \sigma_R \cdot \sqrt{n}$$

Two strategies with the same trade statistics (win rate 40%, +2R / −1R, so $\sigma_R \approx 1.47$) and the same 1% risk, one with 50 trades a year, one with 200.

In [15]:
def simulate_trades(trades_per_year, risk, win_rate=0.40, win_r=2.0, years=YEARS, seed=3):
    rng = np.random.default_rng(seed)
    n_days = ANN * years
    n_trades = rng.poisson(trades_per_year * years)
    days = rng.integers(0, n_days, n_trades)                       # closing day of each trade
    r_mult = np.where(rng.random(n_trades) < win_rate, win_r, -1.0)
    daily = np.bincount(days, weights=risk * r_mult, minlength=n_days)
    return pd.Series(daily, index=pd.bdate_range("2014-01-01", periods=n_days)), r_mult

rows, curves = {}, {}
for n in (50, 200):
    d, r_mult = simulate_trades(n, 0.01, seed=n)
    r_star = 0.01 * TARGET_VOL / ann_vol(d)
    rows[f"{n} trades/yr"] = {"σ_R": r_mult.std(), "formula vol @1%": 0.01 * r_mult.std() * np.sqrt(n),
                              "measured vol @1%": ann_vol(d), "r* for 10% vol": r_star,
                              "vol @r*": ann_vol(d * r_star / 0.01)}
    curves[f"{n} trades/yr @1%"] = d.cumsum()
    curves[f"{n} trades/yr @r* = {r_star:.2%}"] = (d * r_star / 0.01).cumsum()

plot_lines(curves, "Same risk per trade, different annual risk", ytitle="cumulative return")
pd.DataFrame(rows).T.style.format({"σ_R": "{:.2f}", "formula vol @1%": "{:.1%}", "measured vol @1%": "{:.1%}",
                                   "r* for 10% vol": "{:.2%}", "vol @r*": "{:.1%}"})

,σ_R,formula vol @1%,measured vol @1%,r* for 10% vol,vol @r*
50 trades/yr,1.47,10.4%,10.7%,0.93%,10.0%
200 trades/yr,1.48,20.9%,20.9%,0.48%,10.0%


At 1% risk per trade the two strategies run at about 10% and 21% annual vol. The practical method (measure the vol at 1%, then $r^* = 1\% \cdot \sigma_{target} / \sigma_{1\%}$) brings both to 10%. Here the trades are independent and close on one day, so the formula is almost exact; with overlapping trades and correlated outcomes, trust the measurement, not the formula.

## 7. Case 2: no stop loss, volatility-based sizing

Discrete signal $s \in \{-1, 0, +1\}$. When the signal is on:

$$\text{Notional} = s \cdot \text{Equity} \cdot \frac{\sigma_{target}}{\sigma_{instr}}, \qquad \text{Lots} = \frac{\text{Notional}}{\text{Price} \cdot \text{PointValue}}$$

$\sigma_{instr}$ is the EWMA annualized volatility of the instrument, computed with past data only. We test it on a synthetic price with volatility clustering (GARCH) and a simple momentum signal with a flat zone. Two versions: **fixed notional** (1× equity) and **vol-scaled** notional.

In [16]:
def garch_returns(n, omega=2e-6, alpha=0.08, beta=0.90, seed=5):
    rng = np.random.default_rng(seed)
    r, var = np.empty(n), omega / (1 - alpha - beta)
    for t in range(n):
        r[t] = np.sqrt(var) * rng.standard_normal()
        var = omega + alpha * r[t] ** 2 + beta * var
    return r

px_ret = pd.Series(garch_returns(ANN * YEARS), index=raw.index)
mom = px_ret.rolling(50).sum()
signal = np.sign(mom).where(mom.abs() > 0.5 * px_ret.rolling(50).std() * np.sqrt(50), 0).fillna(0)
sigma_instr = (px_ret.pow(2).ewm(span=30).mean() * ANN).pow(0.5).shift(1)

fixed = (signal.shift(1) * px_ret).fillna(0)
scaled = (signal.shift(1) * TARGET_VOL / sigma_instr * px_ret).fillna(0)
f = (signal != 0).mean()

roll = lambda r: r.rolling(63).std() * np.sqrt(ANN)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Instrument volatility (annualized)", "Rolling 3-month volatility of the strategy"))
fig.add_scatter(x=sigma_instr.index, y=sigma_instr, name="σ instrument", line_width=3, row=1, col=1)
fig.add_scatter(x=fixed.index, y=roll(fixed), name="fixed notional", line_width=3, row=2, col=1)
fig.add_scatter(x=scaled.index, y=roll(scaled), name="vol-scaled notional", line_width=3, row=2, col=1)
fig.add_hline(y=TARGET_VOL, line_dash="dash", annotation_text="target", row=2, col=1)
fig.add_hline(y=TARGET_VOL * np.sqrt(f), line_dash="dot", annotation_text="target·√f", row=2, col=1)
fig.update_layout(height=800, title="Vol-scaled sizing keeps the risk stable").show()

print(f"exposure f = {f:.0%}, target·√f = {TARGET_VOL * np.sqrt(f):.1%}")
pd.DataFrame({"fixed notional": metrics(fixed), "vol-scaled": metrics(scaled),
              "vol-scaled, rescaled": metrics(scaled * TARGET_VOL / ann_vol(scaled))}).T.style.format(FMT)

exposure f = 67%, target·√f = 8.2%


,Return,Vol,Sharpe,MaxDD
fixed notional,-0.9%,13.1%,-0.06,-45.6%
vol-scaled,-1.1%,8.3%,-0.13,-40.1%
"vol-scaled, rescaled",-1.3%,10.0%,-0.13,-48.2%


The signal has no edge (the price is a driftless random process): here we only look at the risk. The fixed-notional strategy inherits every volatility regime of the instrument. The vol-scaled one stays in a narrow band, but **below** the target: it is in the market only a fraction $f$ of the days, so its realized vol is about $\sigma_{target}\sqrt{f}$. The last step is the same as in case 1: measure the realized vol and rescale.

Note: equity is the account (or the capital allocated to the strategy); notional is the output of the formula. With 100k equity, 10% target and an instrument at 20% vol, the notional is 50k.

## 8. Weighting methods

All methods run on the vol-targeted returns, long only, weights summing to 1:

| Method | Uses |
| --- | --- |
| Equal weight (1/N) | nothing |
| Inverse volatility | σ |
| Equal risk contribution (ERC) | Σ |
| Minimum variance | Σ |
| Max Sharpe | μ, Σ |
| HRP (López de Prado) | Σ, clustering |

Handcrafting (fixed buckets by strategy type) is in section 13.

In [17]:
def w_equal(r):
    return pd.Series(1 / r.shape[1], r.columns)

def w_inv_vol(r):
    iv = 1 / r.std()
    return iv / iv.sum()

def _long_only(objective, n):
    res = minimize(objective, np.full(n, 1 / n), bounds=[(0, 1)] * n, method="SLSQP",
                   constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
    return res.x

def w_min_var(r):
    cov = r.cov().values
    return pd.Series(_long_only(lambda w: w @ cov @ w * 1e4, r.shape[1]), r.columns)

def w_max_sharpe(r):
    cov, mu = r.cov().values, r.mean().values
    return pd.Series(_long_only(lambda w: -(mu @ w) / np.sqrt(w @ cov @ w), r.shape[1]), r.columns)

def w_erc(r):
    cov = r.cov().values * 1e4
    n = r.shape[1]
    res = minimize(lambda x: 0.5 * x @ cov @ x - np.log(x).sum() / n, np.full(n, 1 / n),
                   bounds=[(1e-8, None)] * n, method="L-BFGS-B")
    return pd.Series(res.x / res.x.sum(), r.columns)

def w_hrp(r):
    cov, corr = r.cov().values, r.corr().values
    dist = np.sqrt(np.clip((1 - corr) / 2, 0, None))
    order = list(leaves_list(linkage(squareform(dist, checks=False), "single")))
    w = np.ones(len(order))
    clusters = [order]
    while clusters:
        clusters = [c[i:j] for c in clusters for i, j in ((0, len(c) // 2), (len(c) // 2, len(c))) if len(c) > 1]
        for a, b in zip(clusters[::2], clusters[1::2]):
            var = [cl_w @ cov[np.ix_(cl, cl)] @ cl_w for cl in (a, b)
                   for cl_w in [(1 / np.diag(cov)[cl]) / (1 / np.diag(cov)[cl]).sum()]]
            alpha = 1 - var[0] / (var[0] + var[1])
            w[a] *= alpha
            w[b] *= 1 - alpha
    return pd.Series(w, r.columns)

METHODS = {"1/N": w_equal, "Inverse vol": w_inv_vol, "ERC": w_erc,
           "Min variance": w_min_var, "Max Sharpe": w_max_sharpe, "HRP": w_hrp}

weights_full = pd.DataFrame({m: f(vt) for m, f in METHODS.items()})
plot_bars(weights_full * 100, "Weights on the full sample (vol-targeted strategies)", ytitle="weight (%)")
weights_full.style.format("{:.1%}")

,1/N,Inverse vol,ERC,Min variance,Max Sharpe,HRP
TF_US500,8.3%,8.3%,6.8%,0.8%,4.5%,6.4%
TF_XAUUSD,8.3%,8.3%,8.3%,10.2%,16.4%,6.4%
TF_GBPJPY,8.3%,8.3%,7.5%,5.6%,0.0%,8.9%
TF_WTI,8.3%,8.3%,7.8%,7.1%,2.8%,8.6%
TF_CORN,8.3%,8.3%,8.0%,9.0%,2.0%,8.9%
MR_US500,8.3%,8.3%,8.4%,8.4%,2.5%,8.9%
MR_NAS100,8.3%,8.3%,8.8%,9.5%,26.9%,7.2%
MR_EURUSD,8.3%,8.3%,9.1%,9.9%,0.0%,8.7%
MR_GER40,8.3%,8.3%,9.0%,9.7%,11.4%,7.2%
SEAS_US500,8.3%,8.3%,8.2%,8.6%,13.7%,9.4%


Inverse volatility is identical to 1/N after vol targeting. ERC and HRP stay close to 1/N, overweighting the strategies with low correlation to the rest. Minimum variance is more uneven (from 1% to 12.5%) and uses only the covariance. Max Sharpe concentrates: MR_NAS100 takes 27%, SEAS_NATGAS 20%, and three strategies get zero. The true Sharpe ratios do differ here (0.24 to 0.94), but max Sharpe sees the **measured** ones, which are noisy: for example it gives 16% to TF_XAUUSD, the strategy with the lowest true Sharpe, because it happened to do well in this sample.

**Where each method sits.** Annualized return and volatility of every method's weights, in sample on the full history, against the long-only efficient frontier (grey dots: the single strategies; the right panel zooms on the cluster of methods that sit almost on the same point). Max Sharpe is the tangent portfolio *by construction*, but the frontier is drawn with estimated means: whether it is the right portfolio out of sample is the question of the next sections.

In [18]:
mu, cov = vt.mean().values * ANN, vt.cov().values * ANN
n_s = len(mu)
def frontier_vol(target):
    res = minimize(lambda w: w @ cov @ w * 1e4, np.full(n_s, 1 / n_s), bounds=[(0, 1)] * n_s, method="SLSQP",
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}, {"type": "eq", "fun": lambda w: mu @ w - target}])
    return np.sqrt(res.fun / 1e4)
w_mv = weights_full["Min variance"].values
grid = np.linspace(mu @ w_mv, mu.max(), 150)          # efficient part only: from the minimum variance portfolio upwards
front = pd.Series(grid * 100, index=[frontier_vol(t) * 100 for t in grid])
colors = {"1/N": "#EF553B", "Inverse vol": "#FFA15A", "ERC": "#00CC96", "Min variance": "#AB63FA",
          "Max Sharpe": "#FFD92F", "HRP": "#19D3F3"}
symbols = {"Inverse vol": "circle-open"}

fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.08,
                    subplot_titles=("Full view", "Zoom on the cluster"))
for col in (1, 2):
    fig.add_scatter(x=front.index, y=front, mode="lines", line=dict(width=5, color="#636EFA"),
                    name="efficient frontier (long only)", legendgroup="frontier", showlegend=col == 1, row=1, col=col)
fig.add_scatter(x=vt.std() * np.sqrt(ANN) * 100, y=mu * 100, mode="markers+text", text=vt.columns, textposition="middle right",
                textfont=dict(size=10, color="grey"), marker=dict(size=7, color="grey", opacity=0.7),
                name="single strategies", row=1, col=1)
for col in (1, 2):
    for m in weights_full:
        w = weights_full[m].values
        fig.add_scatter(x=[np.sqrt(w @ cov @ w) * 100], y=[mu @ w * 100], mode="markers", name=m, legendgroup=m,
                        showlegend=col == 1, hovertemplate=f"{m}<br>vol %{{x:.2f}}%<br>return %{{y:.2f}}%<extra></extra>",
                        marker=dict(size=22 if m == "Inverse vol" else 16, symbol=symbols.get(m, "diamond"), color=colors[m],
                                    line=dict(width=2, color="white" if m != "Inverse vol" else colors[m])), row=1, col=col)
fig.update_xaxes(range=[3, 11.5], title_text="annualized volatility (%)", row=1, col=1)
fig.update_yaxes(range=[0, 15], title_text="annualized return (%)", row=1, col=1)
fig.update_xaxes(range=[4.60, 4.76], title_text="annualized volatility (%)", row=1, col=2)
fig.update_yaxes(range=[7.25, 7.62], title_text="annualized return (%)", row=1, col=2)
fig.add_shape(type="rect", x0=4.60, x1=4.76, y0=7.25, y1=7.62, line=dict(color="grey", dash="dot"), row=1, col=1)
fig.update_layout(title="Risk and return of the weighting methods (in sample)", height=700, legend_itemsizing="constant")
fig.show()

1/N, inverse vol, ERC, HRP and minimum variance sit almost on the same point (about 7.5% return at 4.7% vol): with these strategies, vol-targeted and diversified, the long-only frontier is flat around the equal-weight portfolio. Max Sharpe is the only one that moves away, to 10.3% return at 5.2% vol (in-sample Sharpe 1.96 against 1.57): it takes the extra return from the strategies with the best *measured* Sharpe. That is exactly the part that does not survive out of sample.

## 9. Equal weight + correlation

Equal weight on vol-targeted strategies, scaled so that the **portfolio** runs at the target volatility:

$$w = \frac{1}{\sqrt{N\,(1 + (N-1)\,\bar\rho)}} = \frac{DM}{N}$$

The function below is the one in the presentation: it returns the final multiplier per strategy (vol targeting × portfolio weight), to apply to the risk per trade or to the position size.

In [19]:
def portfolio_scaling(rets: pd.DataFrame, target_vol=0.10, n_min=10, ann=252):
    vol = rets.std() * np.sqrt(ann)
    k = target_vol / vol                                  # vol targeting
    n = rets.shape[1]
    corr = rets.corr().values
    rho = max(corr[np.triu_indices(n, 1)].mean(), 0) if n > 1 else 0
    n_eff = max(n, n_min)
    w = 1 / np.sqrt(n_eff * (1 + (n_eff - 1) * rho))      # equal weight + correlation
    return k * w                                          # final multiplier per strategy

mult = portfolio_scaling(raw, TARGET_VOL)
port_formula = (raw * mult).sum(axis=1)
port_1n = vt.mean(axis=1)

print(f"N = {raw.shape[1]}, average correlation = {rho_bar:.3f}, w = {mult.iloc[0] / k.iloc[0]:.3f} (1/N = {1 / raw.shape[1]:.3f})")
example = 0.005 * mult.iloc[0] / k.iloc[0]
print(f"risk per trade 0.50% standalone -> {example:.3%} in the portfolio")

plot_lines({"1/N (sum of weights = 1)": port_1n.cumsum(), "equal weight + correlation": port_formula.cumsum()},
           "Same portfolio, different leverage", ytitle="cumulative return")
pd.DataFrame({"1/N": metrics(port_1n), "EW + correlation": metrics(port_formula)}).T.style.format(FMT)

N = 12, average correlation = 0.153, w = 0.176 (1/N = 0.083)
risk per trade 0.50% standalone -> 0.088% in the portfolio


,Return,Vol,Sharpe,MaxDD
1/N,7.4%,4.7%,1.57,-6.6%
EW + correlation,15.7%,10.0%,1.57,-14.0%


Same Sharpe ratio (1.57, the weights are proportional), but plain 1/N runs well below the 10% target, because the strategies are not perfectly correlated. The formula brings the portfolio back to the target: each strategy trades at a larger fraction of its standalone size than $1/N$.

Example from the slide: $N = 10$, $\bar\rho = 0.2$ → $w = 0.189$.

In [20]:
print(f"N = 10, rho = 0.2 -> w = {1 / np.sqrt(10 * (1 + 9 * 0.2)):.3f}")

N = 10, rho = 0.2 -> w = 0.189


## 10. Why optimizing the portfolio overfits

The noisiest input of any optimizer is the expected return. The standard error of a Sharpe ratio estimated on $T$ years:

$$SE(SR) \approx \sqrt{\frac{1 + SR^2/2}{T}}$$

In [21]:
def sr_se(sr, years):
    return np.sqrt((1 + sr ** 2 / 2) / years)

print(f"SR 0.5 over 10 years: SE = {sr_se(0.5, 10):.2f}")
print(f"SR 0.6 over 3 years (one WFA in-sample window): SE = {sr_se(0.6, 3):.2f}")

yearly = vt.groupby(vt.index.year).apply(lambda d: d.mean() / d.std() * np.sqrt(ANN))
fig = go.Figure()
for c in yearly:
    fig.add_scatter(x=yearly.index, y=yearly[c], mode="markers", marker_size=11, name=c)
fig.add_scatter(x=[yearly.index[0], yearly.index[-1]], y=[STRATS.sr.mean()] * 2, mode="lines", line_dash="dash", name="average true Sharpe")
fig.update_layout(title="One-year Sharpe ratios: true edges between 0.2 and 1.0, huge dispersion", xaxis_title="year",
                  yaxis_title="Sharpe ratio", height=700).show()
print(f"std of (one-year Sharpe - true Sharpe) across strategies and years: {yearly.sub(STRATS.sr).stack().std():.2f} (theory: {sr_se(STRATS.sr.mean(), 1):.2f})")

SR 0.5 over 10 years: SE = 0.34
SR 0.6 over 3 years (one WFA in-sample window): SE = 0.63


std of (one-year Sharpe - true Sharpe) across strategies and years: 1.04 (theory: 1.08)


True Sharpe ratios are between 0.24 and 0.94 (average 0.59). In any single year the estimates spread from -1.9 to +4.5, and the standard deviation of the estimation error (1.04) matches the theory (1.08): it is much larger than the differences between the strategies. An optimizer fed with 1–3 years of data cannot tell skill from luck: it puts large weights on whatever was lucky in its window. Below, the max Sharpe weights estimated on each 3-year window.

In [22]:
IS_YEARS, OOS_YEARS = 3, 1
starts = range(0, len(vt) - ANN * IS_YEARS + 1, ANN)
ms_weights = pd.DataFrame({vt.index[s + ANN * IS_YEARS - 1].year: w_max_sharpe(vt.iloc[s: s + ANN * IS_YEARS])
                           for s in starts}).T
fig = go.Figure([go.Bar(x=ms_weights.index.astype(str), y=ms_weights[c] * 100, name=c) for c in ms_weights])
fig.update_layout(barmode="stack", title="Max Sharpe weights, re-estimated on rolling 3-year windows",
                  xaxis_title="end of in-sample window", yaxis_title="weight (%)", height=700).show()

The allocation jumps from one window to the next: the largest weight goes to MR_NAS100, SEAS_US500, SEAS_NATGAS or MR_GER40 depending on the window (from 22% to 51% of the portfolio). On average the weights lean towards the strategies with the higher true Sharpe (rank correlation about 0.5), so the optimizer is not blind, but each single estimate is dominated by luck.

Do the weights follow the true edge? Average max Sharpe weight over the windows against the **true** Sharpe of each strategy (known only because the data is synthetic), with the 1/N weight as reference.

In [23]:
avg_w = ms_weights.mean()
fig = go.Figure(go.Scatter(x=STRATS.sr, y=avg_w * 100, mode="markers+text", text=STRATS.index, textposition="top center",
                           marker_size=12))
fig.add_hline(y=100 / len(STRATS), line_dash="dash", annotation_text="1/N weight")
fig.update_layout(title="Average max Sharpe weight vs true Sharpe", xaxis_title="true Sharpe ratio",
                  yaxis_title="average weight (%)", height=650)
fig.show()
print(f"rank correlation weight vs true Sharpe: {avg_w.corr(STRATS.sr, method='spearman'):.2f}")

rank correlation weight vs true Sharpe: 0.48


The weights are correlated with the true Sharpe (rank correlation 0.48), but loosely: TF_US500 (true Sharpe 0.75) gets 4% while SEAS_NATGAS (0.61) gets 19%, and TF_XAUUSD, the strategy with the lowest true Sharpe (0.24), gets 14%, well above 1/N (8.3%). The optimizer picks up part of the real signal and a lot of noise.

## 11. Walk-forward on the portfolio optimization

Same idea as a strategy walk-forward, applied to the weights:

1. in-sample window (3 years): estimate the weights with the method under test;
2. out-of-sample window (next year): apply those weights **unchanged**;
3. roll forward one year, concatenate the OOS returns;
4. compare every method with **1/N on the same OOS periods**.

Every portfolio is scaled to 10% vol with the in-sample covariance (for 1/N this is exactly the equal weight + correlation formula), so the curves are comparable.

Metrics: OOS Sharpe, in-sample Sharpe, WFE = SR_OOS / SR_IS, turnover (average sum of absolute weight changes between windows).

In [24]:
def walk_forward(rets, method, is_len=ANN * IS_YEARS, oos_len=ANN * OOS_YEARS, target_vol=TARGET_VOL):
    oos, is_sr, ws = [], [], []
    for s in range(0, len(rets) - is_len - oos_len + 1, oos_len):
        ins, out = rets.iloc[s: s + is_len], rets.iloc[s + is_len: s + is_len + oos_len]
        w = method(ins)
        lev = target_vol / (np.sqrt(w @ ins.cov().values @ w) * np.sqrt(ANN))
        oos.append(out @ w * lev)
        is_sr.append(sharpe(ins @ w))
        ws.append(w)
    ws = pd.DataFrame(ws)
    return pd.concat(oos), np.mean(is_sr), ws.diff().abs().sum(axis=1).iloc[1:].mean()

wfa, wfa_rows = {}, {}
for m, f in METHODS.items():
    r_oos, sr_is, turnover = walk_forward(vt, f)
    wfa[m] = r_oos
    wfa_rows[m] = {"Sharpe IS": sr_is, "Sharpe OOS": sharpe(r_oos), "WFE": sharpe(r_oos) / sr_is,
                   "Vol OOS": ann_vol(r_oos), "MaxDD OOS": max_dd(r_oos), "Turnover": turnover}
wfa = pd.DataFrame(wfa)
wfa_table = pd.DataFrame(wfa_rows).T
wfa_table["ΔSharpe vs 1/N"] = wfa_table["Sharpe OOS"] - wfa_table.loc["1/N", "Sharpe OOS"]

plot_lines({c: wfa[c].cumsum() for c in wfa}, "Walk-forward of the allocation methods: out-of-sample only",
           ytitle="cumulative return")
wfa_table.style.format({"Sharpe IS": "{:.2f}", "Sharpe OOS": "{:.2f}", "WFE": "{:.2f}", "Vol OOS": "{:.1%}",
                        "MaxDD OOS": "{:.1%}", "Turnover": "{:.2f}", "ΔSharpe vs 1/N": "{:+.2f}"})

,Sharpe IS,Sharpe OOS,WFE,Vol OOS,MaxDD OOS,Turnover,ΔSharpe vs 1/N
1/N,1.76,1.73,0.99,10.4%,-15.3%,0.00,+0.00
Inverse vol,1.77,1.76,1.00,10.5%,-15.3%,0.04,+0.03
ERC,1.78,1.80,1.01,10.5%,-15.3%,0.06,+0.07
Min variance,1.82,1.93,1.06,10.8%,-14.9%,0.23,+0.20
Max Sharpe,2.54,1.50,0.59,10.8%,-14.5%,0.72,-0.23
HRP,1.75,1.82,1.04,10.5%,-15.0%,0.17,+0.09


**Is the difference real?** Block bootstrap of the paired OOS daily returns (blocks of 20 days, so the autocorrelation and the cross-correlation are kept): distribution of $SR_{method} - SR_{1/N}$ and the share of resamples where the method does **not** beat 1/N.

In [25]:
def bootstrap_delta_sr(a, b, n_boot=2000, block=20, seed=11):
    rng = np.random.default_rng(seed)
    x = np.column_stack([a.values, b.values])
    n = len(x)
    n_blocks = int(np.ceil(n / block))
    starts = rng.integers(0, n - block, (n_boot, n_blocks))
    idx = (starts[:, :, None] + np.arange(block)).reshape(n_boot, -1)[:, :n]
    s = x[idx]
    sr = s.mean(axis=1) / s.std(axis=1) * np.sqrt(ANN)
    return sr[:, 0] - sr[:, 1]

boot = {m: bootstrap_delta_sr(wfa[m], wfa["1/N"]) for m in METHODS if m not in ("1/N", "Inverse vol")}
fig = go.Figure([go.Histogram(x=d, name=m, opacity=0.55, nbinsx=80) for m, d in boot.items()])
fig.add_vline(x=0, line_dash="dash")
fig.update_layout(barmode="overlay", title="Bootstrap of ΔSharpe vs 1/N (OOS)", xaxis_title="ΔSharpe", height=650).show()
pd.DataFrame({m: {"ΔSharpe": sharpe(wfa[m]) - sharpe(wfa["1/N"]), "5%": np.percentile(d, 5),
                  "95%": np.percentile(d, 95), "P(Δ ≤ 0)": (d <= 0).mean()} for m, d in boot.items()}).T.round(3)

,ΔSharpe,5%,95%,P(Δ ≤ 0)
ERC,0.070,0.031,0.106,0.002
Min variance,0.196,0.062,0.321,0.010
Max Sharpe,-0.233,-0.564,0.100,0.880
HRP,0.089,0.016,0.155,0.020


**Is the advantage stable over time?** Rolling 1-year Sharpe of each method minus the 1/N Sharpe on the same days (out of sample only), and the distribution of the Sharpe of each calendar year.

In [26]:
roll = {m: wfa[m].rolling(ANN).apply(lambda x: x.mean() / x.std() * np.sqrt(ANN), raw=True) for m in ("ERC", "Min variance", "Max Sharpe", "HRP", "1/N")}
delta_roll = pd.DataFrame({m: roll[m] - roll["1/N"] for m in ("ERC", "Min variance", "Max Sharpe", "HRP")}).dropna()
fig = go.Figure([go.Scatter(x=delta_roll.index, y=delta_roll[m], name=m, line_width=3) for m in delta_roll])
fig.add_hline(y=0, line_dash="dash")
fig.update_layout(title="Rolling 1-year Sharpe minus 1/N (out of sample)", yaxis_title="ΔSharpe", height=600, hovermode="x unified")
fig.show()
print("share of days with ΔSharpe > 0:", (delta_roll > 0).mean().round(2).to_dict())

share of days with ΔSharpe > 0: {'ERC': 0.89, 'Min variance': 0.86, 'Max Sharpe': 0.39, 'HRP': 0.81}


ERC, minimum variance and HRP stay above 1/N on 81–89% of the days: the advantage is small but steady, not the result of one lucky year. Max Sharpe is below 1/N on 61% of the days, with large swings in both directions.

In [27]:
yearly_oos = wfa.groupby(wfa.index.year).apply(lambda d: d.mean() / d.std() * np.sqrt(ANN))
fig = go.Figure([go.Box(y=yearly_oos[m], name=m, boxpoints="all", jitter=0.4, pointpos=0) for m in yearly_oos])
fig.update_layout(title="Out-of-sample Sharpe by calendar year", yaxis_title="Sharpe ratio", height=650, showlegend=False)
fig.show()
print("years where the method beats 1/N:", {m: int((yearly_oos[m] > yearly_oos["1/N"]).sum()) for m in yearly_oos if m != "1/N"}, "of", len(yearly_oos))

years where the method beats 1/N: {'Inverse vol': 9, 'ERC': 9, 'Min variance': 9, 'Max Sharpe': 4, 'HRP': 8} of 10


By calendar year (the first and last are partial, because the out-of-sample blocks are 252 days) ERC and minimum variance beat 1/N in 9 years out of 10 and HRP in 8; max Sharpe in 4, with large losses against 1/N in some years (0.38 against 2.72 in 2019) and gains in others (4.01 against 3.45 in 2021): the average hides a very irregular path.

Reading the results:

- **Max Sharpe** has the best in-sample Sharpe (2.54) and the worst out of sample (1.50): WFE 0.59, turnover 0.72 (the weights change by 72% every year). It is 0.23 below 1/N, but the difference is **not significant**: the 90% interval is -0.56 to +0.10 and the bootstrap puts it below 1/N in 88% of the resamples. Here the true Sharpe ratios really differ, so there is something to find; the estimation error and the turnover eat all of it, and a bit more.
- **Covariance-only methods** (ERC, HRP, minimum variance) beat 1/N by 0.07 to 0.20, and the difference is significant (at most 2% of the resamples are below 1/N). The correlation *level* moves over time (section 15), but the correlation *structure* is persistent: strategies on the same family, instrument or asset class stay more correlated than the others. Three years of daily data are enough to estimate it (not the means), and these methods exploit it. The synthetic design is probably kinder than real life here: on real strategies the structure is less stable, so the gain is usually smaller, and the walk-forward on your own OOS returns is how you find out.
- **Inverse vol** differs slightly from 1/N only because, inside each window, the vols are not exactly 10%.
- The lesson is not "never optimize": it is that estimating **expected returns** is noisy and costs turnover, while using correlations can help if the walk-forward confirms it. And testing five methods against 1/N is itself multiple testing.

## 12. The hidden trap: in-sample strategy returns

The walk-forward of the weights is only honest if the **strategy returns** it receives are out of sample too. Experiment: 40 candidate strategies with **zero edge** (true Sharpe 0). We build a portfolio of 10 of them in two ways:

- **backtest-selected**: pick the 10 best on the full-sample backtest, then run the portfolio walk-forward with 1/N. This is what happens when the inputs are in-sample backtests;
- **honest**: at each walk-forward step, select the 10 best using only the in-sample window, then trade them in the next year.

In [28]:
cand_meta = pd.DataFrame({"family": "other", "instrument": [f"X{i}" for i in range(40)], "asset_class": "none",
                          "win_rate": 0.5, "exposure": 0.6, "raw_vol": TARGET_VOL}, index=[f"S{i}" for i in range(40)])
cand, _ = make_strategies(cand_meta, sr=1e-6, stress=(), seed=7, exact=False)   # true Sharpe = 0

best_full = cand.columns[np.argsort(-sharpe(cand).values)[:10]]
picked, _, _ = walk_forward(cand[best_full], w_equal)

def select_then_equal(ins, k=10):
    w = pd.Series(0.0, ins.columns)
    w[ins.columns[np.argsort(-sharpe(ins).values)[:k]]] = 1 / k
    return w

honest, _, _ = walk_forward(cand, select_then_equal)
plot_lines({"backtest-selected (in-sample inputs)": picked.cumsum(), "selected in-sample only (honest)": honest.cumsum()},
           "40 strategies with zero edge: the same walk-forward, two kinds of input", ytitle="cumulative return")
pd.DataFrame({"backtest-selected": metrics(picked), "honest": metrics(honest)}).T.style.format(FMT)

,Return,Vol,Sharpe,MaxDD
backtest-selected,8.5%,10.5%,0.81,-18.0%
honest,0.3%,10.4%,0.03,-37.6%


None of the 40 strategies has an edge, yet the backtest-selected portfolio shows an out-of-sample Sharpe of 0.81: the selection used the full sample, which contains the "out-of-sample" years. The portfolio walk-forward is formally correct, its inputs are not. With the selection done in-sample only, the result is about zero (0.03), as it should be.

The same happens with strategies whose parameters were optimized on the full history. Feed the portfolio walk-forward with the OOS segments of each strategy's own walk-forward (or live returns), and include each strategy only from the date it actually existed.

## 13. Balancing strategy types

1/N gives equal weight to **strategies**, not to **ideas**: the family with more strategies gets more risk. Fixed buckets decided a priori (here 40% trend following, 40% mean reversion, 20% other), equal weight inside each bucket, then the whole portfolio scaled to the target vol.

In [29]:
BUCKETS = {"trend": 0.40, "meanrev": 0.40, "other": 0.20}

def w_buckets(r, meta=STRATS, buckets=BUCKETS):
    fam = meta.loc[r.columns, "family"]
    return fam.map(buckets) / fam.map(fam.value_counts())

def rc_by(w, rets, groups):
    return pd.Series(risk_contrib(w.values, rets.cov().values), w.index).groupby(groups).sum()

fam = STRATS.family
rc_fam = pd.DataFrame({"1/N": rc_by(w_equal(vt), vt, fam), "buckets 40/40/20": rc_by(w_buckets(vt), vt, fam)})
plot_bars(rc_fam * 100, "Risk contribution by strategy type", ytitle="% of portfolio risk")

r_buckets, sr_is_b, _ = walk_forward(vt, w_buckets)
print(f"WFA OOS Sharpe: 1/N {sharpe(wfa['1/N']):.2f} | buckets {sharpe(r_buckets):.2f}")
rc_fam.style.format("{:.0%}")

WFA OOS Sharpe: 1/N 1.73 | buckets 1.74


,1/N,buckets 40/40/20
family,,
meanrev,30%,40%
other,23%,17%
trend,47%,43%


With 5 trend, 4 mean reversion and 3 other strategies, 1/N puts almost half of the risk (47%) on trend following, also because trend strategies are more correlated with each other. The buckets are chosen by reasoning, not estimated: there is nothing to overfit. Note that equal **weight** per bucket is not equal **risk** per bucket: correlation within a family raises its share. Check the risk contributions, not only the weights.

## 14. Balancing instruments and asset classes

Same check by instrument and by asset class. Five of our twelve strategies trade equity indices, three of them on the US500.

In [30]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("by instrument", "by asset class"))
for col, g in enumerate(["instrument", "asset_class"], 1):
    rc_g = rc_by(w_equal(vt), vt, STRATS[g]).sort_values(ascending=False)
    fig.add_bar(x=rc_g.index, y=rc_g * 100, name=g, row=1, col=col)
fig.add_hline(y=25, line_dash="dash", annotation_text="cap 25%", row=1, col=2)
fig.update_layout(title="Risk contribution of the 1/N portfolio", yaxis_title="% of portfolio risk",
                  height=650, showlegend=False).show()

us = STRATS.index[STRATS.instrument == "US500"]
c_us = vt[us].corr().values
print(f"average correlation among the {len(us)} US500 strategies: {c_us[np.triu_indices(len(us), 1)].mean():.2f} "
      f"(portfolio average {rho_bar:.2f})")

average correlation among the 3 US500 strategies: 0.29 (portfolio average 0.15)


Indices carry the largest share of the risk: more strategies, and higher correlation between strategies on the same instrument. The fix is a cap per instrument/asset class, and a development priority: the next strategy should fill an underweight asset class, not add a fourth one on the US500.

## 15. Correlation is not stable

Rolling 3-month average pairwise correlation of the twelve strategies. Besides the slow drift of the baseline (the yearly average goes from 0.02 to 0.26), the shaded areas are the three stress periods built into the data (average correlation 0.30 inside, 0.15 outside): the rolling estimate reacts only as the stress days enter the window, and stays high after the stress has ended.

In [31]:
def rolling_avg_corr(r, window=63):
    iu = np.triu_indices(r.shape[1], 1)
    vals = [np.corrcoef(r.iloc[i - window: i].values.T)[iu].mean() for i in range(window, len(r) + 1)]
    return pd.Series(vals, index=r.index[window - 1:])

rc_roll = rolling_avg_corr(vt)
stress_blocks = stress.astype(int).diff().fillna(0)
fig = go.Figure(go.Scatter(x=rc_roll.index, y=rc_roll, name="3-month average correlation", line_width=4))
for a, b in zip(stress.index[stress_blocks == 1], stress.index[stress_blocks == -1]):
    fig.add_vrect(x0=a, x1=b, fillcolor="red", opacity=0.2, line_width=0)
fig.add_hline(y=rho_bar, line_dash="dash", annotation_text="full-sample average")
fig.update_layout(title="Average correlation between strategies over time", yaxis_title="correlation", height=650).show()

What it means for the portfolio vol: the formula sets the leverage with one correlation number. If the correlation rises, the realized vol overshoots the target. Rolling 1-month realized vol of the portfolio, with the leverage set from the full-sample average correlation and from a conservative estimate (90th percentile of the rolling correlation), plus the safeguards: correlation floored at 0, DM capped at 2.5.

In [32]:
def ew_weight(n, rho, dm_cap=2.5):
    rho = max(rho, 0)                                  # no credit for negative correlation
    return min(1 / np.sqrt(n * (1 + (n - 1) * rho)), dm_cap / n)

n = vt.shape[1]
rho_cons = rc_roll.quantile(0.9)
port_avg = vt.sum(axis=1) * ew_weight(n, rho_bar)
port_cons = vt.sum(axis=1) * ew_weight(n, rho_cons)
rv = lambda r: r.rolling(21).std() * np.sqrt(ANN)

fig = go.Figure()
fig.add_scatter(x=port_avg.index, y=rv(port_avg), name=f"ρ = average ({rho_bar:.2f})", line_width=3)
fig.add_scatter(x=port_cons.index, y=rv(port_cons), name=f"ρ = 90th pct ({rho_cons:.2f})", line_width=3)
for a, b in zip(stress.index[stress_blocks == 1], stress.index[stress_blocks == -1]):
    fig.add_vrect(x0=a, x1=b, fillcolor="red", opacity=0.2, line_width=0)
fig.add_hline(y=TARGET_VOL, line_dash="dash", annotation_text="target")
fig.update_layout(title="Portfolio realized vol (1-month): correlation spikes break the target",
                  yaxis_title="annualized vol", height=650).show()

pd.DataFrame({"ρ used": [rho_bar, rho_cons], "DM": [n * ew_weight(n, rho_bar), n * ew_weight(n, rho_cons)],
              "vol normal": [ann_vol(p[~stress]) for p in (port_avg, port_cons)],
              "vol stress": [ann_vol(p[stress]) for p in (port_avg, port_cons)]},
             index=["average ρ", "conservative ρ"]).round(3)

,ρ used,DM,vol normal,vol stress
average ρ,0.153,2.115,0.095,0.157
conservative ρ,0.285,1.702,0.076,0.126


In the stress periods the realized vol jumps from 9.5% to 15.7% with the average correlation, well above the 10% target. The conservative estimate gives up vol in normal times (7.6%) to limit the overshoot when it matters (12.6%).

## 16. Practical issues: minimum N and lot rounding

**Minimum N.** With few strategies live, the formula gives each one a large share of risk, without the diversification that justifies it. With `n_min = 10` the portfolio simply runs below target until it is full.

In [33]:
n_live = np.arange(1, 21)
rho_ex = 0.2
w_free = np.array([ew_weight(n, rho_ex, dm_cap=np.inf) for n in n_live])
w_min = np.array([ew_weight(max(n, 10), rho_ex, dm_cap=np.inf) for n in n_live])
port_vol = lambda w, n: TARGET_VOL * w * np.sqrt(n * (1 + (n - 1) * rho_ex))

fig = make_subplots(rows=1, cols=2, subplot_titles=("weight per strategy", "portfolio vol"))
for w, name, color in ((w_free, "N = live count", "#636efa"), (w_min, "N_eff = max(N, 10)", "#00cc96")):
    fig.add_scatter(x=n_live, y=w, name=name, line=dict(width=4, color=color), row=1, col=1)
    fig.add_scatter(x=n_live, y=port_vol(w, n_live), name=name, line=dict(width=4, color=color), row=1, col=2, showlegend=False)
fig.update_xaxes(title_text="strategies live")
fig.update_layout(title=f"Minimum N (ρ = {rho_ex})", height=600).show()

**Lot rounding.** Risk per trade in money = equity × standalone risk per trade × $w$. The lot size is that amount divided by the money lost at the stop with 1.00 lot, rounded **down** to the lot step (0.01). Illustrative instrument: the stop of 1.00 lot costs on average €1,000, varying from trade to trade (ATR-based stop, lognormal). Standalone risk per trade 0.5%, $\bar\rho = 0.2$.

In [ ]:
def lot_rounding(equity, n, risk=0.005, rho=0.2, stop_cost=1000, stop_disp=0.4, lot_step=0.01, n_trades=20_000, seed=9):
    rng = np.random.default_rng(seed)
    cost = stop_cost * rng.lognormal(-stop_disp ** 2 / 2, stop_disp, n_trades)       # € lost at the stop per 1.00 lot
    target = equity * risk * ew_weight(max(n, 10), rho, dm_cap=np.inf)
    lots = np.floor(target / cost / lot_step) * lot_step
    taken = lots > 0
    return pd.Series({"risk per trade €": target, "skipped": 1 - taken.mean(),
                      "actual / target risk (taken)": (lots[taken] * cost[taken]).mean() / target if taken.any() else np.nan})

grid = {(eq, n): lot_rounding(eq, n) for eq in (10_000, 100_000) for n in (10, 15, 20, 30)}
lots_table = pd.DataFrame(grid).T
lots_table.index.names = ["equity", "N"]

fig = go.Figure([go.Bar(x=[str(n) for n in (10, 15, 20, 30)], y=lots_table.loc[eq, "skipped"] * 100, name=f"equity {eq:,}")
                 for eq in (10_000, 100_000)])
fig.update_layout(title="Trades skipped because the lot rounds to zero", xaxis_title="N strategies",
                  yaxis_title="% of trades skipped", barmode="group", height=600).show()
lots_table.style.format({"risk per trade €": "{:,.1f}", "skipped": "{:.0%}", "actual / target risk (taken)": "{:.0%}"})

- **100k account**: no trade is skipped, but rounding down leaves the actual risk at 86–95% of the target, and the gap grows with N.
- **10k account**: with 10 strategies the target risk per trade is about €9, below the €10 risk of the minimum lot on an average stop: about half the trades are skipped; with 30 strategies almost all of them.
- The skipped trades are not random: they are the ones with the widest stops, that is the high-volatility periods. Live results then differ from the backtest in a systematic way.
- Options: fewer strategies on small accounts, more equity per strategy, instruments with smaller contracts, or rounding to the nearest lot (accepting risk above target on small positions).

## 17. Choosing the target volatility

For a prop firm account the target vol comes from the rules. Monte Carlo with a block bootstrap of the portfolio's daily returns (equal weight + correlation, normalized to each target vol). Illustrative rules: profit target +10%, maximum daily loss 5%, maximum loss 10% (both on the initial balance), 120 trading days. Daily closes only: intraday drawdowns are not modeled, so the daily-loss risk is underestimated.

In [35]:
def challenge_mc(r, target_vol, n_paths=5000, days=120, block=10, target=0.10, daily_limit=0.05, max_loss=0.10, seed=21):
    rng = np.random.default_rng(seed)
    x = (r / ann_vol(r) * target_vol).values
    n_blocks = int(np.ceil(days / block))
    idx = (rng.integers(0, len(x) - block, (n_paths, n_blocks))[:, :, None] + np.arange(block)).reshape(n_paths, -1)[:, :days]
    d = x[idx]
    eq = d.cumsum(axis=1)
    hit_t = np.where((eq >= target).any(1), (eq >= target).argmax(1), days)
    fail_daily_t = np.where((d <= -daily_limit).any(1), (d <= -daily_limit).argmax(1), days)
    fail_max_t = np.where((eq <= -max_loss).any(1), (eq <= -max_loss).argmax(1), days)
    first = np.vstack([hit_t, fail_daily_t, fail_max_t]).argmin(0)
    ended = np.minimum(np.minimum(hit_t, fail_daily_t), fail_max_t) < days
    return pd.Series({"pass": ((first == 0) & ended).mean(), "fail: daily loss": ((first == 1) & ended).mean(),
                      "fail: max loss": ((first == 2) & ended).mean(), "still running": (~ended).mean()})

vols = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
mc = pd.DataFrame({f"{v:.0%}": challenge_mc(port_formula, v) for v in vols}).T
colors = {"pass": "#2ca02c", "fail: daily loss": "#ff7f0e", "fail: max loss": "#d62728", "still running": "#7f7f7f"}
fig = go.Figure([go.Bar(x=mc.index, y=mc[c] * 100, name=c, marker_color=colors[c]) for c in mc])
fig.update_layout(barmode="stack", title="Challenge outcomes vs target volatility (120 days)",
                  xaxis_title="annual target vol", yaxis_title="% of paths", height=650).show()
print(f"daily vol: 10% annual -> {0.10 / np.sqrt(ANN):.2%}, 20% annual -> {0.20 / np.sqrt(ANN):.2%}")
mc.style.format("{:.1%}")

daily vol: 10% annual -> 0.63%, 20% annual -> 1.26%


,pass,fail: daily loss,fail: max loss,still running
5%,4.3%,0.0%,0.0%,95.7%
10%,46.5%,0.0%,1.5%,52.0%
15%,68.0%,0.0%,6.7%,25.3%
20%,72.1%,8.9%,9.8%,9.1%
25%,69.5%,17.2%,10.8%,2.5%
30%,67.4%,21.6%,10.4%,0.6%
40%,61.1%,30.6%,8.2%,0.1%


- At 10% vol the portfolio passes in 47% of the paths, and half are still running after 120 days: too slow.
- The pass rate peaks around 20% vol (72%) and stays near 70% up to 30%. Daily-loss breaches appear at 20% (9%) and grow quickly above (17% at 25%, 22% at 30%).
- Pick the target from the expected value of an attempt (fee, payout, probability of failing), not from the highest pass rate.
- The portfolio here has a Sharpe of about 1.6, optimistic for real strategies: with a lower Sharpe every pass rate drops. The rules are illustrative: replace them with the real ones of the account.

## Takeaways

- Diversification raises the Sharpe ratio, but the average correlation sets a ceiling; most of the gain comes from the first 10–15 genuinely different strategies.
- Vol-target every strategy before weighting: rescale the risk per trade (stop loss) or size the notional on the instrument's vol (no stop loss), then measure and rescale.
- Equal weight + correlation keeps the portfolio at the target vol with nothing to estimate except one correlation number.
- Expected returns are mostly noise to an optimizer: any allocation method must beat 1/N out of sample in a walk-forward, with a significant margin. Methods that use only the covariance can help, if the walk-forward confirms it.
- Feed the portfolio walk-forward with out-of-sample strategy returns, or the result is selection bias.
- Balance strategy types and instruments with buckets and caps decided a priori, and check risk contributions, not only weights.
- Correlations rise under stress: floor ρ at 0, cap the DM, consider a conservative ρ, set a minimum N, and check lot rounding at your account size.